# 03 — Spatial Visualization

**Pipeline Step 3 of 5**

This notebook visualizes Stabl-selected biomarker gene expression via **spatial overlay** (when spatial coordinates are available) or **UMAP embedding** (fallback). A bonus UMAP colored by condition (WT vs AD) is also generated to confirm separation. Prox1 (Hippocampus Dentate Gyrus marker) is always included for anatomical verification.

### Inputs
| File | Description |
|---|---|
| `data/processed/ad_preprocessed.h5ad` | QC-filtered, normalized AnnData from Step 01 |
| `cache/stabl_results_<hash>.pkl` | Stabl results from Step 02 |

### Outputs
| File | Description |
|---|---|
| `assets/umap_<Gene>.png` / `assets/spatial_<Gene>.png` | Expression plots for top 5 markers + Prox1 + Trem2 + Gfap |
| `assets/umap_condition.png` | UMAP colored by WT/AD condition |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import load_adata, run_stabl_cached, plot_spatial_markers

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"
ASSETS_DIR = PROJECT_ROOT / "assets"

print("Imports ready.")

## 3.1 Load Data and Stabl Results

Reload the preprocessed AD AnnData and cached Stabl results. Sort markers by descending stability score.

In [ ]:
import pandas as pd

adata = load_adata(DATA_PROCESSED / "ad_preprocessed.h5ad")

stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="geo_ad",
    label_method="condition",
    n_bootstraps=500,
    prefilter="de",
    fdr_alpha=0.01,
    min_log2fc=0.5,
)

df_features = pd.DataFrame({
    "Gene": stabl_result["selected_genes"],
    "Stability Score": [
        stabl_result["stability_scores"][g]
        for g in stabl_result["selected_genes"]
    ],
}).sort_values("Stability Score", ascending=False)
top_markers = df_features["Gene"].tolist()

print(f"Loaded {adata.shape[0]} spots, {stabl_result['n_selected']} Stabl markers")

## 3.2 Generate Marker Plots

For each of the top 5 markers, generates a spatial overlay or UMAP plot (fallback when spatial coordinates lack full image metadata). A condition UMAP is also generated.

In [ ]:
saved_plots = plot_spatial_markers(
    adata,
    markers=top_markers,
    save_dir=ASSETS_DIR,
    n_top=5,
)

print(f"\n{len(saved_plots)} spatial plots saved to {ASSETS_DIR}")

## 3.3 Display Plots

In [ ]:
from IPython.display import Image, display

for plot_path in saved_plots:
    display(Image(filename=str(plot_path), width=500))